# Quantum Canary demo

Quantum Canary checks the health of qubits in one short job.

- **IBM (superconducting):** 74 probe circuits per qubit (8,640 shots) measure T1, T2 (echo),
  T2\* (Ramsey), the frequency offset Δω and the single-qubit gate error ε, each with an error bar,
  a fit check (χ²/dof) and a GREEN / YELLOW / RED verdict.
- **IonQ (trapped ions):** T1 and T2 last seconds, so only ε is measured. The report, saving,
  comparing and monitoring work exactly the same way.

Contents:
1. Settings
2. Check qubits and read the results
3. Use the results in your own code
4. Save and load
5. Compare two checks
6. The command line
7. IonQ trapped ions
8. No hardware: simulate, with the true values

By default everything runs on IBM's **fake** backends: free, local, no account. Each check takes
about five minutes because your laptop simulates it.

## Install

From a clone of the repository, in a terminal with your virtual environment active:
```
python -m pip install -e .
quantum-canary --version
```

In [1]:
import warnings
warnings.filterwarnings("ignore")

import math
import pathlib
import subprocess
import sys

import quantum_canary as canary
canary.__version__

'0.9.0'

## 1. Settings

Edit this cell. Real hardware uses QPU time or credits, so both switches start off.

- `USE_IBM = True` runs on real IBM hardware through your saved account
  (`QiskitRuntimeService.save_account(...)`, once) or the token files named below.
- `RUN_IONQ = True` runs section 7 on IonQ. It needs the `IONQ_API_KEY` environment variable or the token file named below.
  `IONQ_BACKEND = "simulator"` is IonQ's free cloud simulator (with the Forte-1 noise model);
  `"qpu.forte-1"` is the real machine and costs credits.

In [2]:
USE_IBM = False
IBM_BACKEND = "ibm_kingston"      # the fake copy is used when USE_IBM is False
QUBITS = [13, 14]                 # neighbours on Kingston, so they are probed in two groups

# Secrets kept in files are used only if you name the file here. None means the vendors' usual
# lookup: your saved IBM account, and the IONQ_API_KEY environment variable.
IBM_TOKEN_FILE = None             # e.g. "ibm_api_token.txt"
IBM_INSTANCE_FILE = None          # e.g. "ibm_crn_instance.txt"
IONQ_TOKEN_FILE = None            # e.g. "ionq_api_token.txt"

RUN_IONQ = False
IONQ_BACKEND = "simulator"
IONS = [0, 3]

OUT = pathlib.Path("demo_output")  # where this notebook saves its files
OUT.mkdir(exist_ok=True)

In [3]:
if USE_IBM:
    service = canary.credentials.ibm_service(IBM_TOKEN_FILE, IBM_INSTANCE_FILE)
    backend = service.backend(IBM_BACKEND)
else:
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    backend = FakeProviderForBackendV2().backend("fake_" + IBM_BACKEND.removeprefix("ibm_"))

backend.name, backend.num_qubits

('fake_kingston', 156)

## 2. Check qubits and read the results

`canary.check(backend, qubits)` sends one job, fits the counts and returns a **Report**.

In [4]:
r = canary.check(backend, QUBITS)
r.print()

qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
13    396 ± 19 us      413 ± 29 us      441 ± 27 us      -0.00901 ± 0.026 2.45e-04 ± 3e-05    ok                        GREEN
14    282 ± 14 us      286 ± 21 us      286 ± 19 us      -0.0642 ± 0.041  2.09e-04 ± 3e-05    ok                        GREEN

dynamical decoupling: not needed on these qubits

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does, that probe's
        numbers may be wrong.
Health  GREEN   every fit converged and passed, and T2* is within 20% of T2
        YELLOW  a fit failed its limit, or T2* differs from T2 by more than 20%
                (slow frequency noise, which dynamical decoupling refocuses)
        RED

How to read the table:

- Each number is `value ± error bar`. Repeated checks scatter by about one error bar.
- **fit**: `ok`, or which probe fit poorly (e.g. `T1 6.2 > 4.4`). Only that probe's numbers are in doubt.
- **health**: GREEN = trust the numbers; YELLOW = a fit was poor or T2\* and T2 differ by more than
  20% (the bracket says which); RED = a fit failed, re-run.
- Health describes the measurement, not the qubit's quality.

Each qubit's results as plain numbers. Times are in seconds and Δω in rad/s:

In [5]:
q = r[QUBITS[0]]
print(f"T1   = {q.t1 * 1e6:6.1f} ± {q.t1_sigma * 1e6:.1f} µs")
print(f"T2   = {q.t2 * 1e6:6.1f} ± {q.t2_sigma * 1e6:.1f} µs")
print(f"T2*  = {q.t2_star * 1e6:6.1f} ± {q.t2_star_sigma * 1e6:.1f} µs")
print(f"dw   = {q.delta_omega / (2 * math.pi):6.1f} ± {q.delta_omega_sigma / (2 * math.pi):.1f} Hz")
print(f"eps  = {q.eps:.2e} ± {q.eps_sigma:.0e}")
print("health:", q.health, "|", q.health_reason or "-", "| poor fits:", q.fit)
print("advice:", q.dd_advice, q.frame_advice)

T1   =  396.4 ± 18.7 µs
T2   =  413.0 ± 29.4 µs
T2*  =  441.0 ± 27.1 µs
dw   =   -9.0 ± 25.5 Hz
eps  = 2.45e-04 ± 3e-05
health: GREEN | - | poor fits: []
advice: [] []


## 3. Use the results in your own code

Canary only reports; you decide.

In [6]:
trusted = [q.qubit for q in r if q.health == "GREEN"]
retry = [q.qubit for q in r if q.health == "RED"]
use_dd = any(q.dd_advice for q in r)

print("trust:", trusted, "| re-run:", retry, "| turn on dynamical decoupling:", use_dd)
for q in r:
    if q.health != "GREEN":
        print(f"qubit {q.qubit} is {q.health}: {q.health_reason} {q.fit}")

trust: [13, 14] | re-run: [] | turn on dynamical decoupling: False


## 4. Save and load

The file holds every number, the priors used and the raw shot counts. `canary.load` gives the
Report back; files from the command line load the same way.

In [7]:
r.save(OUT / "before.json")
before = canary.load(OUT / "before.json")
print(before[QUBITS[0]].t1 == r[QUBITS[0]].t1)

True


## 5. Compare two checks

`canary.compare(before, after)` asks, per qubit and per number, how many combined error bars the
two checks are apart (z), and flags a change only past a limit corrected for testing four numbers
at once. A qubit that did not change is flagged about 5% of the time.

In [8]:
after = canary.check(backend, QUBITS)
after.save(OUT / "after.json")

d = canary.compare(before, after)
d.print()
print("changed:", d.changed, "| stable:", d.stable)

qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    GREEN    YELLOW       -1.41    +1.28    +0.68    -0.58   -
14    GREEN    GREEN        +1.10    +0.37    +0.95    -0.61   -

changed = |z| > 2.00 after a Holm correction over each qubit's four parameters; z uses both checks' error bars
stable: nothing changed beyond the error bars
changed: [] | stable: True


## 6. The command line

The same two operations from a terminal:
```
quantum-canary check fake_kingston 13,14 --json a.json
quantum-canary compare a.json b.json
quantum-canary compare first.json second.json third.json     # every later file vs the first
```
Here they run through the notebook's Python (`cli` prints the output with the right encoding).

In [9]:
def cli(*args):
    out = subprocess.run([sys.executable, "-m", "quantum_canary", *map(str, args)],
                         capture_output=True, text=True, encoding="utf-8")
    print(out.stderr + out.stdout)

qubits_arg = ",".join(map(str, QUBITS))
cli("check", backend.name, qubits_arg, "--json", OUT / "cli.json")

c:\Users\Kishu\OneDrive\GitHub\quantum-canary\.venv\Lib\site-packages\qiskit_ibm_runtime\fake_provider\backends\nighthawk\fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(
checking qubits [13, 14] on fake_kingston (local fake backend) ...
saved demo_output\cli.json
qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
13    341 ± 16 us      421 ± 30 us      425 ± 26 us      -0.053 ± 0.025   2.20e-04 ± 2e-05    ok                        GREEN
14    244 ± 12 us      264 ± 20 us      336 ± 22 us      -0.0155 ± 0.036  2.28e-04 ± 3e-05    ok                        YELLOW (T2*/T2 1.27)

dynamical decoupling: not needed on these qubits

Fit     chi2/dof of each probe against its limit. On a qubit that fol

In [10]:
cli("compare", OUT / "before.json", OUT / "after.json", OUT / "cli.json")

baseline demo_output\before.json
2 checks against the baseline: changed = |z| > 2.28, so that the chance of any false alarm over all of them is that of one check at |z| > 2

demo_output\after.json
qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    GREEN    YELLOW       -1.41    +1.28    +0.68    -0.58   -
14    GREEN    GREEN        +1.10    +0.37    +0.95    -0.61   -

changed = |z| > 2.28 after a Holm correction over each qubit's four parameters; z uses both checks' error bars
stable: nothing changed beyond the error bars

demo_output\cli.json
qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    GREEN    GREEN        -2.26    +0.19    -1.22    -0.76   -
14    GREEN    YELLOW       -2.08    -0.77    +0.89    +0.41   -

changed = |z| > 2.28 after a Holm correction over each qubit's four parameters

## 7. IonQ trapped ions

The same `canary.check` on a qiskit-ionq backend measures ε on each ion, one ion per circuit.
Only ε is measured, so T1, T2, T2\* and Δω show as `n/a`, and health uses the gate fit alone.
From a terminal: `quantum-canary check qpu.forte-1 0,3` (credentials as in section 1).

`canary.check` has no cost limit. On the real machine use
`canary.run_ionq(backend, ions, max_cost=...)`, which refuses a run whose IonQ estimate is higher.

In [11]:
if RUN_IONQ:
    provider = canary.credentials.ionq_provider(IONQ_TOKEN_FILE)
    ion_backend = provider.get_backend(IONQ_BACKEND, gateset="native")
    if IONQ_BACKEND == "simulator":
        ion_backend.set_options(noise_model="forte-1")

    ions_before = canary.check(ion_backend, IONS)
    ions_before.print()
    ions_before.save(OUT / "ions_before.json")

    ions_after = canary.check(ion_backend, IONS)
    canary.compare(ions_before, ions_after).print()
else:
    print("RUN_IONQ is False. Set it in section 1 to run this on IonQ.")

RUN_IONQ is False. Set it in section 1 to run this on IonQ.


## 8. No hardware: simulate, with the true values

`canary.simulate` invents qubits with hidden true values, measures them with the same probe and
returns the same Report, plus `q.truth`. Same seed, same qubits. `regime="ideal"` follows Canary's
model exactly; `regime="nisq"` (default) adds drift, slow frequency noise and coherent gate error.

In [12]:
sc = canary.simulate(2, architecture="superconducting", regime="nisq")
sc.print()

qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
0     303 ± 17 us      82.9 ± 6.3 us    90.8 ± 5.8 us    +0.164 ± 0.13    3.75e-04 ± 3e-05    ok                        GREEN
1     110 ± 5.2 us     95.4 ± 6.8 us    65.2 ± 4.6 us    +14.1 ± 0.18     5.12e-04 ± 5e-05    ok                        YELLOW (T2*/T2 0.68)

dynamical decoupling advised on qubit 1: T2*/T2 = 0.68 < 0.8
frequency offset on qubit 1: |Δω|·T2* = 5.8 rad > 1; correct it with a virtual-Z frame update or frequency recalibration (DD only if it drifts)

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does, that probe's
        numbers may be wrong.
Health  GREEN   every fit converged and passed, and T2* is within 20% of 

In [13]:
pairs = [("t1", "T1_s"), ("t2", "T2_s"), ("delta_omega", "delta_omega"), ("eps", "epsilon_sx")]

def within_two_sigma(report):
    for q in report:
        for name, key in pairs:
            value, sigma, truth = getattr(q, name), getattr(q, name + "_sigma"), q.truth[key]
            print(f"qubit {q.qubit} {name:12s} {value:10.3g}  true {truth:10.3g}  "
                  f"within 2 error bars: {abs(value - truth) < 2 * sigma}")

within_two_sigma(sc)

qubit 0 t1             0.000303  true    0.00028  within 2 error bars: True
qubit 0 t2             8.29e-05  true   9.23e-05  within 2 error bars: True
qubit 0 delta_omega    1.03e+03  true   1.01e+03  within 2 error bars: True
qubit 0 eps            0.000375  true   0.000324  within 2 error bars: True
qubit 1 t1              0.00011  true   0.000115  within 2 error bars: True
qubit 1 t2             9.54e-05  true   0.000104  within 2 error bars: True
qubit 1 delta_omega    8.88e+04  true    8.8e+04  within 2 error bars: True
qubit 1 eps            0.000512  true   0.000498  within 2 error bars: True


The true ε is the error per pulse that randomized benchmarking measures on the simulated qubit,
coherent gate error included; Canary's gate probe is randomized benchmarking, so it measures the same.

Trapped ions, simulated (all five numbers, since a simulation can measure T1 and T2):

In [14]:
ti = canary.simulate(2, architecture="trapped_ion", regime="nisq")
ti.print()

qubit T1               T2               T2*              dw/2pi (Hz)      eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
0     1.29e+05 ± 6.1e+03 ms473 ± 34 ms      390 ± 25 ms      -0.145 ± 0.03    2.17e-03 ± 2e-04    ok                        GREEN
1     1.84e+05 ± 8.2e+03 ms1.33e+03 ± 1e+02 ms808 ± 54 ms      -0.0941 ± 0.017  2.11e-03 ± 2e-04    ok                        YELLOW (T2*/T2 0.61)

dynamical decoupling advised on qubit 1: T2*/T2 = 0.61 < 0.8

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does, that probe's
        numbers may be wrong.
Health  GREEN   every fit converged and passed, and T2* is within 20% of T2
        YELLOW  a fit failed its limit, or T2* differs from T2 by more than 20%
                (slow frequency noise, which dynami